# HelpVisual ML server (v4)

Runs everything the HelpVisual app needs on Kaggle's free GPUs, with no outside AI service:

- **GPU 0:** Stable Diffusion XL with a canny ControlNet (images), and EasyOCR (text and its position on screenshots).
- **GPU 1:** Qwen2.5-7B-Instruct, 4-bit (steps and answers), CLIP ViT-L/14 (image checks), and NLLB-200 1.3B (translation into Hindi, Telugu, Spanish and French).

NLLB-200 is licensed CC-BY-NC 4.0: fine for this project, not for commercial use.

**Before running, in the Settings menu:** Accelerator **GPU T4 x2**, Internet **On**.

Then **Run All**. When the last cell prints `READY`, paste the lines it shows into the backend terminal on your laptop.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.device_count() >= 2, "Two GPUs are needed: set Accelerator to GPU T4 x2, then run again."
!pip install -q diffusers transformers accelerate safetensors bitsandbytes easyocr fastapi uvicorn sentencepiece

In [ ]:
%%writefile server.py
"""HelpVisual ML server, running on a Kaggle GPU notebook.

GPU 0: Stable Diffusion XL (image generation), its canny ControlNet, and EasyOCR.
GPU 1: an open language model (OpenAI-compatible chat endpoint), CLIP ViT-L/14 (image checks) and
NLLB-200 (translation into Hindi, Telugu, Spanish and French).

Long requests stream whitespace until the result is ready. JSON parsers ignore leading
whitespace, and the steady bytes stop the tunnel closing a request that runs over 100 seconds.
"""
import base64
import hmac
import io
import json
import os
import threading
import time
import uuid
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import TimeoutError as FutureTimeout

import easyocr
import numpy as np
import torch
from diffusers import (
    AutoencoderKL,
    ControlNetModel,
    DPMSolverMultistepScheduler,
    StableDiffusionXLControlNetPipeline,
    StableDiffusionXLPipeline,
)
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import StreamingResponse
from PIL import Image
from pydantic import BaseModel, Field
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    CLIPModel,
    CLIPProcessor,
)

TOKEN = os.environ["HV_TOKEN"]
SDXL_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
CLIP_MODEL = "openai/clip-vit-large-patch14"
CONTROLNET_MODEL = "diffusers/controlnet-canny-sdxl-1.0"
LLM_MODEL = os.environ.get("HV_LLM", "Qwen/Qwen2.5-7B-Instruct")
TRANSLATOR_MODEL = "facebook/nllb-200-distilled-1.3B"  # licence: CC-BY-NC 4.0 (non-commercial)
NLLB_CODES = {"en": "eng_Latn", "hi": "hin_Deva", "te": "tel_Telu", "es": "spa_Latn", "fr": "fra_Latn"}

if torch.cuda.device_count() < 2:
    raise SystemExit("This server needs two GPUs. Set Accelerator to GPU T4 x2 and run again.")
GEN_DEVICE, AUX_DEVICE = "cuda:0", "cuda:1"

# ---------- image generation (GPU 0) ----------
# The fp16-safe VAE avoids the black images the stock SDXL VAE can produce in half precision.
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained(
    SDXL_MODEL, vae=vae, torch_dtype=torch.float16, variant="fp16", use_safetensors=True
).to(GEN_DEVICE)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe.set_progress_bar_config(disable=True)
try:
    controlnet = ControlNetModel.from_pretrained(CONTROLNET_MODEL, torch_dtype=torch.float16, variant="fp16")
except (OSError, ValueError):  # no fp16 files published: load the full weights in half precision
    controlnet = ControlNetModel.from_pretrained(CONTROLNET_MODEL, torch_dtype=torch.float16)
# The ControlNet pipeline shares SDXL's weights, so it adds only the ControlNet's memory.
control_pipe = StableDiffusionXLControlNetPipeline(
    vae=pipe.vae,
    text_encoder=pipe.text_encoder,
    text_encoder_2=pipe.text_encoder_2,
    tokenizer=pipe.tokenizer,
    tokenizer_2=pipe.tokenizer_2,
    unet=pipe.unet,
    controlnet=controlnet.to(GEN_DEVICE),
    scheduler=pipe.scheduler,
)
control_pipe.set_progress_bar_config(disable=True)
reader = easyocr.Reader(["en"], gpu=True)

# ---------- image checks and language model (GPU 1) ----------
clip = CLIPModel.from_pretrained(CLIP_MODEL, torch_dtype=torch.float16).to(AUX_DEVICE).eval()
processor = CLIPProcessor.from_pretrained(CLIP_MODEL)
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
    ),
    device_map={"": AUX_DEVICE},
).eval()

nllb_tokenizer = AutoTokenizer.from_pretrained(TRANSLATOR_MODEL)
nllb = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATOR_MODEL, torch_dtype=torch.float16).to(AUX_DEVICE).eval()

gen_lock, aux_lock = threading.Lock(), threading.Lock()
executor = ThreadPoolExecutor(max_workers=4)
app = FastAPI(title="HelpVisual ML server")


def check_token(x_token: str | None, authorization: str | None) -> None:
    supplied = x_token or (authorization or "").removeprefix("Bearer ").strip()
    if not supplied or not hmac.compare_digest(supplied, TOKEN):
        raise HTTPException(401, "Invalid token.")


def keepalive_json(work):
    """Runs work() in a thread, sending a space every 5 s until its JSON result is ready."""
    future = executor.submit(work)

    def stream():
        while True:
            try:
                result = future.result(timeout=5)
            except FutureTimeout:
                yield b" "
                continue
            except Exception as exc:  # reported in the body: the 200 status is already sent
                yield json.dumps({"error": {"message": str(exc)[:500]}}).encode()
                return
            yield json.dumps(result).encode()
            return

    return StreamingResponse(stream(), media_type="application/json")


def clip_similarities(images: list[Image.Image], texts: list[str]) -> torch.Tensor:
    """Cosine similarity matrix, images x texts."""
    inputs = processor(text=texts, images=images, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(AUX_DEVICE) for k, v in inputs.items()}
    inputs["pixel_values"] = inputs["pixel_values"].half()
    with aux_lock, torch.no_grad():
        output = clip(**inputs)
    image_embeds = torch.nn.functional.normalize(output.image_embeds.float(), dim=-1)
    text_embeds = torch.nn.functional.normalize(output.text_embeds.float(), dim=-1)
    return image_embeds @ text_embeds.T


def to_png_b64(image: Image.Image) -> str:
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode()


class GenerateRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=2000)
    negative_prompt: str = Field(default="", max_length=1000)
    width: int = Field(default=1152, ge=512, le=1344, multiple_of=8)
    height: int = Field(default=896, ge=512, le=1344, multiple_of=8)
    seed: int = Field(default=0, ge=0)
    steps: int = Field(default=30, ge=10, le=50)
    candidates: int = Field(default=2, ge=1, le=4)
    select_text: str = Field(default="", max_length=1000)  # what the chosen candidate should match
    control_image_b64: str | None = None  # edge map for the ControlNet, same size as the image
    control_scale: float = Field(default=0.85, ge=0.0, le=1.5)


class VerifyRequest(BaseModel):
    image_b64: str
    texts: list[str] = Field(min_length=1, max_length=40)
    target_index: int = 0


class TranslateRequest(BaseModel):
    texts: list[str] = Field(min_length=1, max_length=200)
    source: str
    target: str


class OcrRequest(BaseModel):
    image_b64: str
    min_confidence: float = Field(default=0.3, ge=0.0, le=1.0)


class ChatRequest(BaseModel):
    model: str = ""
    messages: list[dict] = Field(min_length=1)
    temperature: float = 0.2
    max_tokens: int | None = None
    response_format: dict | None = None


@app.get("/health")
def health() -> dict:
    return {"status": "ready", "image_model": SDXL_MODEL, "clip_model": CLIP_MODEL, "llm": LLM_MODEL,
            "controlnet": CONTROLNET_MODEL, "translator": TRANSLATOR_MODEL, "ocr_layout": True}


@app.post("/generate")
def generate(body: GenerateRequest, x_token: str | None = Header(default=None), authorization: str | None = Header(default=None)):
    check_token(x_token, authorization)
    control = None
    if body.control_image_b64:
        try:
            control = Image.open(io.BytesIO(base64.b64decode(body.control_image_b64))).convert("RGB")
        except Exception as exc:
            raise HTTPException(422, "control_image_b64 is not a valid image.") from exc
        if control.size != (body.width, body.height):
            control = control.resize((body.width, body.height))

    def work() -> dict:
        started = time.time()
        images = []
        common = dict(
            prompt=body.prompt,
            negative_prompt=body.negative_prompt or None,
            width=body.width,
            height=body.height,
            num_inference_steps=body.steps,
            guidance_scale=6.5,
        )
        with gen_lock:
            for k in range(body.candidates):
                generator = torch.Generator(device=GEN_DEVICE).manual_seed(body.seed + k)
                if control is None:
                    result = pipe(**common, generator=generator)
                else:
                    result = control_pipe(
                        **common, image=control, controlnet_conditioning_scale=body.control_scale, generator=generator
                    )
                images.append(result.images[0])
        scores: list[float] = []
        chosen = 0
        if len(images) > 1:
            sims = clip_similarities(images, [body.select_text or body.prompt])[:, 0]
            scores = [round(float(s), 4) for s in sims]
            chosen = int(torch.argmax(sims))
        return {
            "image_b64": to_png_b64(images[chosen]),
            "scores": scores,
            "chosen": chosen,
            "seconds": round(time.time() - started, 1),
        }

    return keepalive_json(work)


@app.post("/verify")
def verify(body: VerifyRequest, x_token: str | None = Header(default=None), authorization: str | None = Header(default=None)) -> dict:
    check_token(x_token, authorization)
    try:
        image = Image.open(io.BytesIO(base64.b64decode(body.image_b64))).convert("RGB")
    except Exception as exc:
        raise HTTPException(422, "image_b64 is not a valid image.") from exc
    similarities = clip_similarities([image], body.texts)[0].tolist()
    # OCR: confident word-like text in a generated image is a hallucination candidate.
    with gen_lock:
        detections = reader.readtext(np.array(image))
    ocr_text = [text for _, text, confidence in detections if confidence >= 0.5 and sum(c.isalpha() for c in text) >= 3]
    return {"similarities": [round(float(s), 4) for s in similarities], "ocr_text": ocr_text}


@app.post("/v1/chat/completions")
def chat_completions(body: ChatRequest, x_token: str | None = Header(default=None), authorization: str | None = Header(default=None)):
    """OpenAI-compatible chat endpoint for the open language model."""
    check_token(x_token, authorization)
    messages = []
    for message in body.messages:
        content = message.get("content", "")
        if isinstance(content, list):
            if any(part.get("type") != "text" for part in content):
                raise HTTPException(400, "This server's language model does not accept images.")
            content = "\n".join(part.get("text", "") for part in content)
        messages.append({"role": message.get("role", "user"), "content": content})
    json_mode = (body.response_format or {}).get("type") == "json_object"
    if json_mode:
        messages.insert(0, {"role": "system", "content": "You are a precise assistant. Reply with one valid JSON object only."})

    def work() -> dict:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(AUX_DEVICE)
        sampling = body.temperature > 0.25  # structured output is decoded greedily
        options = {"do_sample": True, "temperature": body.temperature, "top_p": 0.9} if sampling else {"do_sample": False}
        with aux_lock, torch.no_grad():
            output = llm.generate(
                **inputs,
                max_new_tokens=min(body.max_tokens or 4096, 4096),
                pad_token_id=tokenizer.eos_token_id,
                **options,
            )
        reply = tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return {
            "id": f"chatcmpl-{uuid.uuid4().hex[:12]}",
            "object": "chat.completion",
            "model": LLM_MODEL,
            "choices": [{"index": 0, "message": {"role": "assistant", "content": reply}, "finish_reason": "stop"}],
        }

    return keepalive_json(work)


@app.post("/translate")
def translate(body: TranslateRequest, x_token: str | None = Header(default=None), authorization: str | None = Header(default=None)):
    """Batch translation with NLLB-200. Texts are translated one sentence group at a time in batches of 16."""
    check_token(x_token, authorization)
    if body.source not in NLLB_CODES or body.target not in NLLB_CODES:
        raise HTTPException(422, f"Languages must be among {sorted(NLLB_CODES)}.")

    def work() -> dict:
        out: list[str] = []
        nllb_tokenizer.src_lang = NLLB_CODES[body.source]
        target_id = nllb_tokenizer.convert_tokens_to_ids(NLLB_CODES[body.target])
        for start in range(0, len(body.texts), 16):
            batch = body.texts[start:start + 16]
            inputs = nllb_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256)
            inputs = {k: v.to(AUX_DEVICE) for k, v in inputs.items()}
            with aux_lock, torch.no_grad():
                generated = nllb.generate(**inputs, forced_bos_token_id=target_id, max_new_tokens=256, num_beams=4)
            out += nllb_tokenizer.batch_decode(generated, skip_special_tokens=True)
        return {"translations": out}

    return keepalive_json(work)


@app.post("/ocr")
def ocr(body: OcrRequest, x_token: str | None = Header(default=None), authorization: str | None = Header(default=None)) -> dict:
    """Text on an image with its position: each word group with its box [x0, y0, x1, y1] in pixels."""
    check_token(x_token, authorization)
    try:
        image = Image.open(io.BytesIO(base64.b64decode(body.image_b64))).convert("RGB")
    except Exception as exc:
        raise HTTPException(422, "image_b64 is not a valid image.") from exc
    with gen_lock:
        detections = reader.readtext(np.array(image))
    words = []
    for points, text, confidence in detections:
        if confidence < body.min_confidence:
            continue
        xs, ys = [p[0] for p in points], [p[1] for p in points]
        words.append({"text": text, "confidence": round(float(confidence), 3),
                      "box": [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))]})
    return {"width": image.width, "height": image.height, "words": words}


In [ ]:
import os, re, secrets, subprocess, threading, time, urllib.request

LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
token = secrets.token_urlsafe(24)
os.makedirs("/kaggle/temp/hf_cache", exist_ok=True)
log = open("server.log", "w")
server = subprocess.Popen(
    ["python", "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", "8000"],
    env=dict(os.environ, HV_TOKEN=token, HV_LLM=LLM_MODEL, HF_HOME="/kaggle/temp/hf_cache"),
    stdout=log, stderr=subprocess.STDOUT,
)

print("Loading models. The first run downloads about 31 GB, so allow 12 to 20 minutes...")
for _ in range(360):
    if server.poll() is not None:
        print(open("server.log").read()[-4000:])
        raise SystemExit("The server stopped while starting. The log above shows why.")
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=3)
        break
    except Exception:
        time.sleep(5)
else:
    print(open("server.log").read()[-4000:])
    raise SystemExit("The server did not become ready in 30 minutes.")
print("Models loaded.")

if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared"
    )
    os.chmod("cloudflared", 0o755)
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
for line in tunnel.stdout:
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break
if url is None:
    raise SystemExit("The tunnel did not start. Check that Internet is switched on in the notebook settings.")
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

keys = "IMAGE_PROVIDER|ML_ENDPOINT_URL|ML_ENDPOINT_TOKEN|LLM_PROVIDER|LLM_FALLBACK_PROVIDER|OPENAI_BASE_URL|OPENAI_API_KEY|OPENAI_MODEL|LLM_MIN_INTERVAL"
print("\n" + "=" * 78)
print("READY. In the backend terminal on your laptop: press Ctrl+C, paste all these lines,")
print("then start the server again with:  uvicorn app.main:app --reload --port 8000")
print("=" * 78 + "\n")
print(f"(Get-Content .env) -notmatch '^({keys})=' | Set-Content .env")
for line in [
    "IMAGE_PROVIDER=endpoint",
    f"ML_ENDPOINT_URL={url}",
    f"ML_ENDPOINT_TOKEN={token}",
    "LLM_PROVIDER=openai",
    "LLM_FALLBACK_PROVIDER=",
    f"OPENAI_BASE_URL={url}/v1",
    f"OPENAI_API_KEY={token}",
    f"OPENAI_MODEL={LLM_MODEL}",
    "LLM_MIN_INTERVAL=0",
]:
    print(f'Add-Content .env "{line}"')
print("\nKeep this cell running and this browser tab open while you use the app.")
print("The URL and token change every time this cell is started again.\n")

started = time.time()
while server.poll() is None and tunnel.poll() is None:
    time.sleep(300)
    print(f"Still running ({int((time.time() - started) / 60)} min).")
print(open("server.log").read()[-3000:])
raise SystemExit("The server or tunnel stopped. Run this cell again and update .env with the new lines.")
